In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from underthesea import word_tokenize
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
import re

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
train_df = pd.read_csv('vlsp_sentiment_train_cleaned.csv')
test_df = pd.read_csv('vlsp_sentiment_test_cleaned.csv')

# Chuyển label: -1 -> 0, 1 -> 1
train_df['label'] = train_df['Class'].map({-1: 0, 1: 1})
test_df['label'] = test_df['Class'].map({-1: 0, 1: 1})

# LOẠI BỎ NaN (quan trọng)
# Trong Cell 1, sau khi map label:
train_df['label'] = train_df['label'].fillna(0)   # hoặc dropna
train_df = train_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(float)
test_df['label'] = test_df['label'].astype(float)

# In ra số lượng nan còn lại để kiểm tra
print(f"Remaining NaN in train: {train_df['label'].isna().sum()}")
print(f"Remaining NaN in test: {test_df['label'].isna().sum()}")

# Kiểm tra
assert not train_df['label'].isna().any(), "Train still has NaN"
assert not test_df['label'].isna().any(), "Test still has NaN"
print(f"Train size after dropna: {len(train_df)}, Test size: {len(test_df)}")

# Tiền xử lý văn bản
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text, format="text").split()
    return tokens

train_df['tokens'] = train_df['Data'].apply(preprocess_text)
test_df['tokens'] = test_df['Data'].apply(preprocess_text)
print("Sample tokens:", train_df['tokens'].iloc[0])

Using device: cuda
Remaining NaN in train: 0
Remaining NaN in test: 0
Train size after dropna: 5100, Test size: 700
Sample tokens: ['mình', 'đã', 'dùng', 'anywhere', 'thế_hệ', 'đầu_quả', 'là', 'đầy', 'thất_vọng', 'hiện_tại', 'đang', 'vứt_xó', 'giá', 'thì', 'đắt', 'ngốn_pin', 'như', 'ăn_gỏi', 'nặng']


In [3]:
# CELL 2: Build vocabulary và encode
def build_vocab(tokens_list, min_freq=1):
    counter = Counter()
    for tokens in tokens_list:
        counter.update(tokens)
    vocab = ['<pad>', '<unk>']
    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab.append(word)
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    return word2idx, vocab

word2idx, vocab = build_vocab(train_df['tokens'], min_freq=1)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

def encode_sequence(tokens, word2idx):
    unk_idx = word2idx['<unk>']
    return [word2idx.get(token, unk_idx) for token in tokens]

train_df['encoded'] = train_df['tokens'].apply(lambda x: encode_sequence(x, word2idx))
test_df['encoded'] = test_df['tokens'].apply(lambda x: encode_sequence(x, word2idx))

train_df['length'] = train_df['encoded'].apply(len)
test_df['length'] = test_df['encoded'].apply(len)

max_len = int(np.percentile(train_df['length'], 95))
print(f"Max sequence length: {max_len}")

class SentimentDataset(Dataset):
    def __init__(self, encoded, lengths, labels, max_len):
        self.encoded = encoded
        self.lengths = lengths
        self.labels = labels
        self.max_len = max_len
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        seq = self.encoded[idx]
        length = self.lengths[idx]
        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
            length = self.max_len
        return torch.LongTensor(seq), torch.LongTensor([length]), torch.FloatTensor([self.labels[idx]])

def collate_fn(batch):
    seqs, lengths, labels = zip(*batch)
    seqs = torch.stack(seqs)
    lengths = torch.stack(lengths).squeeze(1)
    labels = torch.stack(labels).squeeze(1)
    return seqs, lengths, labels

batch_size = 64
train_dataset = SentimentDataset(train_df['encoded'].tolist(), train_df['length'].tolist(), train_df['label'].tolist(), max_len)
test_dataset = SentimentDataset(test_df['encoded'].tolist(), test_df['length'].tolist(), test_df['label'].tolist(), max_len)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

Vocab size: 11239
Max sequence length: 69


In [4]:
# CELL 3: Định nghĩa model (không sigmoid)
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.bidirectional = bidirectional
        linear_in = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(linear_in, 1)
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        if self.bidirectional:
            h_n = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            h_n = h_n[-1]
        h_n = self.dropout(h_n)
        return self.fc(h_n).squeeze(1)

In [5]:
embedding_dim = 100
hidden_dim = 128
num_layers = 2
dropout = 0.5

# Sanity check: make sure vocab_size covers all token indices used
max_idx = 0
for seqs, lengths, labels in train_loader:
    max_idx = max(max_idx, seqs.max().item())
for seqs, lengths, labels in test_loader:
    max_idx = max(max_idx, seqs.max().item())
print(f"vocab_size = {vocab_size}, max token index in data = {max_idx}")

safe_vocab_size = max(vocab_size, max_idx + 1)
print(f"Using vocab_size = {safe_vocab_size}")

model_std = SentimentLSTM(safe_vocab_size, embedding_dim, hidden_dim, num_layers, dropout, bidirectional=False).to(device)
model_bi = SentimentLSTM(safe_vocab_size, embedding_dim, hidden_dim, num_layers, dropout, bidirectional=True).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer_std = optim.Adam(model_std.parameters(), lr=0.001)
optimizer_bi = optim.Adam(model_bi.parameters(), lr=0.001)
num_epochs = 10

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch_idx, (seqs, lengths, labels) in enumerate(loader):
        seqs = seqs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        
        # Debug: kiểm tra labels
        if torch.isnan(labels).any():
            print(f"Batch {batch_idx}: NaN found in labels!")
            print("Labels:", labels)
            raise ValueError("NaN in labels")
        if (labels < 0).any() or (labels > 1).any():
            print(f"Batch {batch_idx}: labels out of range [0,1]")
            print("Labels:", labels)
            raise ValueError("Labels out of range")
        
        optimizer.zero_grad()
        logits = model(seqs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for seqs, lengths, labels in loader:
            seqs = seqs.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device).float()
            logits = model(seqs, lengths)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return accuracy_score(true_labels, predictions), f1_score(true_labels, predictions)

print("=== Standard LSTM ===")
for epoch in range(num_epochs):
    loss = train_epoch(model_std, train_loader, criterion, optimizer_std)
    acc, f1 = evaluate(model_std, test_loader)
    print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

print("\n=== Bidirectional LSTM ===")
for epoch in range(num_epochs):
    loss = train_epoch(model_bi, train_loader, criterion, optimizer_bi)
    acc, f1 = evaluate(model_bi, test_loader)
    print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

final_acc_std, final_f1_std = evaluate(model_std, test_loader)
final_acc_bi, final_f1_bi = evaluate(model_bi, test_loader)
print("\n" + "="*50)
print("FINAL RESULTS COMPARISON")
print("="*50)
print(f"Standard LSTM (2 layers)     | Accuracy: {final_acc_std:.4f} | F1: {final_f1_std:.4f}")
print(f"Bidirectional LSTM (2 layers) | Accuracy: {final_acc_bi:.4f} | F1: {final_f1_bi:.4f}")

vocab_size = 11239, max token index in data = 11238
Using vocab_size = 11239
=== Standard LSTM ===
Epoch  1 | Loss: 0.6103 | Acc: 0.6357 | F1: 0.5304
Epoch  2 | Loss: 0.5217 | Acc: 0.6829 | F1: 0.6119
Epoch  3 | Loss: 0.4729 | Acc: 0.7014 | F1: 0.7019
Epoch  4 | Loss: 0.4550 | Acc: 0.7271 | F1: 0.6983
Epoch  5 | Loss: 0.4001 | Acc: 0.7300 | F1: 0.6957
Epoch  6 | Loss: 0.3545 | Acc: 0.7271 | F1: 0.6746
Epoch  7 | Loss: 0.3093 | Acc: 0.7329 | F1: 0.6949
Epoch  8 | Loss: 0.2560 | Acc: 0.7329 | F1: 0.6889
Epoch  9 | Loss: 0.2951 | Acc: 0.7357 | F1: 0.6891
Epoch 10 | Loss: 0.2277 | Acc: 0.7543 | F1: 0.7485

=== Bidirectional LSTM ===
Epoch  1 | Loss: 0.5845 | Acc: 0.6714 | F1: 0.5644
Epoch  2 | Loss: 0.4598 | Acc: 0.6914 | F1: 0.5846
Epoch  3 | Loss: 0.3641 | Acc: 0.6829 | F1: 0.5681
Epoch  4 | Loss: 0.2679 | Acc: 0.7214 | F1: 0.6461
Epoch  5 | Loss: 0.1779 | Acc: 0.7086 | F1: 0.6250
Epoch  6 | Loss: 0.1054 | Acc: 0.7514 | F1: 0.7166
Epoch  7 | Loss: 0.0641 | Acc: 0.7100 | F1: 0.6420
Epoch 